# Inherit from PYNQ Default IP

In [25]:
from pynq import DefaultIP, Overlay

class PmodDriver(DefaultIP):
    bindto = ['xilinx.com:hls:pmod_io:1.0']

    def __init__(self, description):
        super().__init__(description)

    def pmod_io_func(self, io_ctrl, io_num):
        self.write(0x10, io_ctrl)
        self.write(0x18, io_num)
        return self.read(0x20)

overlay = Overlay("kv260_pmod_petalinux.bit")

print(type(overlay.pmod_io_0))

<class '__main__.PmodDriver'>


In [ ]:
result = overlay.pmod_io_0.pmod_io_func(0x01, 0x07)
print(f"Return from pmod_io_func: 0x{result:04X}")

# MMIO direct to AXI

In [28]:
base_addr = overlay.ip_dict['pmod_io_0']['phys_addr']
addr_range = overlay.ip_dict['pmod_io_0']['addr_range']

# Instantiate MMIO object
mmio = MMIO(base_addr, addr_range)

# Register offsets
IO_CTRL_OFFSET = 0x10
IO_NUM_OFFSET  = 0x18
PMOD_VAL_OFFSET = 0x20

# === Turn all pins OFF ===
mmio.write(IO_CTRL_OFFSET, 0x0F)   # Command: clear all
mmio.write(IO_NUM_OFFSET, 0x00)
val = mmio.read(PMOD_VAL_OFFSET)
print(f"[ALL OFF] Pmod return = 0x{val:04X}")

[ALL OFF] Pmod return = 0x0000


In [38]:
base_addr, addr_range

(2684420096, 65536)

In [31]:
# === Turn all pins ON ===
mmio.write(IO_CTRL_OFFSET, 0x01)   # Command: custom bitmask
mmio.write(IO_NUM_OFFSET, 0b11111111)    # 0b11111111 (all pins high)
val = mmio.read(PMOD_VAL_OFFSET)
print(f"[ALL ON] Pmod return = 0x{val:04X}")

[ALL ON] Pmod return = 0x00FF


In [36]:
# === Enable only pins 0, 1, 2 ===
mmio.write(IO_CTRL_OFFSET, 0x01)   # Command: custom bitmask
mmio.write(IO_NUM_OFFSET, 0b10100101)
val = mmio.read(PMOD_VAL_OFFSET)
print(f"Pmod return = 0x{val:04X}")

Pmod return = 0x00A5
